In [1]:
import pandas as pd
import re
from pprint import pprint
import numpy as np


cgg_path = r'data/cgg_raw.tsv'
cgg_df_orig = pd.read_csv(cgg_path, sep='\t', dtype=str)


# Cleaning longitude and latititude columns

Standard cleaning


In [4]:
cgg_df['cleaned_lon'] = (cgg_df.Lon
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        ) 

cgg_df['cleaned_lat'] = (cgg_df.Lat
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        )   

Identify unique latitude formats

In [5]:
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats.unique().tolist()

[nan,
 '+12',
 '+12.12',
 '12',
 '12.12',
 "N 12º12.12'",
 "12°12'12''",
 'N12.12',
 '12.12˚N',
 '12.12 N',
 '12°12\'12.12"N',
 '12.12°',
 '12° 12\' 12.12" N',
 '12°12’12”',
 '12° 12.12´S',
 '12°12\'12.12"',
 '12°12‘12‘‘ N',
 '12° 12’ 12.12”',
 '12.12N',
 "12' 12.12''",
 '12 12.12 N',
 '12°12′N',
 'N12',
 '12 12 12.12 N',
 "W12 12' 12.12''",
 '12deg12\'12.12"S',
 "12°12'N",
 "12°12'12''N",
 '12°12\'12"',
 'N 12.12',
 '-12.12',
 '12.12°N',
 '12˚12\'12.12" N',
 "12 12' 12'' N",
 'N12° 12\' 12.12"',
 '12.12° N',
 '12° 12.12',
 '12°12\'12"N',
 '12o12.12',
 "12° 12' 12.12'' N",
 "12° 12'12.12'' N",
 '12° 12’ 12.12” N',
 '12°12’12.12’’N',
 '12°12\'12.12"S',
 '12° 12.12 N',
 '12°12\'12" N',
 '12° 12´12´´N',
 '12°12.12’N',
 '12° 12′ 12″ N']

Identify unique longitude formats

In [6]:
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats.unique().tolist()

[nan,
 '+12.12',
 '-12.12',
 '-12',
 '12.12',
 "E 12º12.12'",
 "12°12'12''",
 'W12.12',
 '-12.12˚W',
 '12.12 W',
 '12°12\'12.12"E',
 '-12.12°',
 '12° 12\' 12.12" W',
 '12°12.12`W',
 '12°12\'12.12"',
 '12°12‘12‘‘ W',
 '12° 12’ 12.12”',
 '12.12W',
 '12\' 12.12"',
 '12 12.12 E',
 "'+12.12",
 '12°12′W',
 'E12',
 '12 12 12.12 W',
 "N12 12' 12.12''",
 '12.12 E',
 '12deg12\'12.12"E',
 'E12.12',
 'E 12.12',
 "12°12'W",
 "12°12'12''E",
 '12',
 '12.12°',
 '12°12\'12"',
 '-12.12°W',
 '12˚12\'12.12" W',
 "12 12' 12'' E",
 'W12° 12\' 12.12"',
 '12.12˚W',
 '12.12˚E',
 '12.12° E',
 '12°12.12',
 '12o12.12',
 '12.12°E',
 '12.12°W',
 '-12.12˚E',
 "12° 12' 12.12'' W",
 "12° 12'12.12'' W",
 "N12° 12' 12.12'' W",
 '12°12\'12.12"W',
 '12°12\'12.12"Ø',
 '12°12\'12"W',
 "12° 12' 12.12'' V",
 '12°12\'12.12"V',
 '12° 12’ 12.12”Ø.',
 '12° 12’ 12.12”Ø',
 '12°12’12.12’’ E',
 '12° 12 Ø',
 "12' 12.12''",
 '12°12\'12" W',
 '12° 12´12´´ W',
 '12°12.12’W',
 '12° 12′ 12″ E']

### Convert to standard characters and symbols
The bad characters were found from the above step

In [7]:
cgg_df['cleaned_lon'] = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘|`", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

cgg_df['cleaned_lat'] = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

Put direction indicators (N, S, -, +) in a seperate column

In [8]:
pd.set_option('future.no_silent_downcasting', True)

cgg_df['lon_direction'] = (cgg_df.cleaned_lon
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')              
                           )
#  Removes the direction from the cleaned_lon column, as now it is no longer needed
cgg_df.cleaned_lon = (cgg_df.cleaned_lon
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

cgg_df['lat_direction'] = (cgg_df.cleaned_lat
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')  # This will make it easier later, when adding the direction back to the coordinate
                           )

#  Removes the direction from the cleaned_lat column, as now it is no longer needed
cgg_df.cleaned_lat = (cgg_df.cleaned_lat
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

Identify unique formats again

In [9]:
from pprint import pprint
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats = pd.Series(lon_formats.dropna().unique())
lat_formats = pd.Series(lat_formats.dropna().unique())

print('======Lon formats======')
print('With degree symbol')
pprint(lon_formats.dropna()[lon_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lon_formats.dropna()[~lon_formats.dropna().str.contains('°')].unique().tolist())

print()
print('======Lat formats======')
print('With degree symbol')
pprint(lat_formats.dropna()[lat_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lat_formats.dropna()[~lat_formats.dropna().str.contains('°')].unique().tolist())

======Lon formats======
With degree symbol
["12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12°12\'12.12"',
 '12° 12\' 12.12"',
 "12°12'",
 '12°12.12',
 '12° 12\'12.12"',
 '12° 12\' 12.12".',
 '12° 12',
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12\' 12.12"',
 '12 12.12',
 "'12.12",
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"']

======Lat formats======
With degree symbol
["12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12°12\'12.12"',
 '12° 12\' 12.12"',
 "12° 12.12'",
 "12°12'",
 '12° 12.12',
 '12°12.12',
 '12° 12\'12.12"',
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12',
 '12.12',
 '12\' 12.12"',
 '12 12.12',
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"']


From the above it seems safe to assume that if:
1. A coordinate contains 1 number with or without traling ° that the format is decimal degrees.
2. A coordinate contains 2 numbers seperated with with either a whitespace or ° or both that its formatted as degrees and minutes.
3. A coordinate contains 3 numbers, where first seperator is by ° or whitespace or both and second seperator is ' or whitespace or both, it is formatted as degree minute seconds.

We can formulate this using the following regular expressions:

In [10]:
dd_regex_lon = r'''^\d{1,3}(\.\d+)?(°| °)?$'''
dm_regex_lon = r'''^\d{1,3}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex_lon = r'''^\d{1,3}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''

dd_regex = r'''^\d{1,2}(\.\d+)?(°| °)?$'''
dm_regex = r'''^\d{1,2}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex = r'''^\d{1,2}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''


Checking the general formats lon

In [11]:
def check_format_lon(s: str):
    if re.match(dd_regex_lon, s):
        return 'DD'
    elif re.match(dm_regex_lon, s):
        return 'DM'
    elif re.match(dms_regex_lon, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lon_formats.apply(check_format_lon)

lon_with_formats = pd.DataFrame({'ExampleLongitude': lon_formats, 'Format': classified_formats}).sort_values(by='Format')
lon_with_formats

,ExampleLongitude,Format
0,12.12,DD
1,12,DD
4,12.12°,DD
2,12°12.12',DM
17,12° 12,DM
14,12°12.12,DM
8,12 12.12,DM
10,12°12',DM
15,"12° 12'12.12""",DMS
13,"12 12' 12""",DMS


Checking the general formats lat

In [12]:
def check_format_lat(s: str):
    if re.match(dd_regex, s):
        return 'DD'
    elif re.match(dm_regex, s):
        return 'DM'
    elif re.match(dms_regex, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lat_formats.apply(check_format_lat)

lat_with_formats = pd.DataFrame({'ExampleLatitude': lat_formats, 'Format': classified_formats}).sort_values(by='Format')
lat_with_formats                           

,ExampleLatitude,Format
0,12,DD
1,12.12,DD
4,12.12°,DD
15,12°12.12,DM
14,12° 12.12,DM
10,12°12',DM
7,12° 12.12',DM
9,12 12.12,DM
2,12°12.12',DM
17,"12° 12'12""",DMS


In [13]:
lat_with_formats.to_excel('data/lat_formats.xlsx', index=False)
lon_with_formats.to_excel('data/lon_formats.xlsx', index=False)

Apply the same format identification to the actual data

In [14]:
cgg_df['lon_format'] = cgg_df.cleaned_lon.map(check_format_lon, na_action='ignore')
cgg_df['lat_format'] = cgg_df.cleaned_lat.map(check_format_lat, na_action='ignore')

Manual check that the lon format identification was done correctly

In [15]:
print('========Lon=========')
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    print(lon_formats)
    print()
    
print()
print('========Lat=========')
    
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    print(lat_formats)
    print()

========Lon=========
nan
Series([], dtype: object)

DD
0       d.dd
1     ddd.dd
2      dd.dd
3     dd.dd°
4    ddd.dd°
5      d.dd°
dtype: object

invalid format
0           dddddddd
1         dd' dd.dd"
2             dddddd
3              'd.dd
4            ddddddd
5         dd°dd'ddd"
6    dd° dd' dd.dd".
7         dd° dddddd
dtype: object

DM
0     dd°dd.dd'
1    ddd°dd.dd'
2      dd dd.dd
3        dd°dd'
4       ddd°dd'
5     ddd°dd.dd
dtype: object

DMS
0           dd°dd'dd"
1        dd°dd'dd.dd"
2       d° dd' dd.dd"
3     ddd° dd' dd.dd"
4         dd dd dd.dd
5      ddd dd' dd.dd"
6        ddd dd dd.dd
7       ddd°dd'dd.dd"
8          dd dd' dd"
9      dd° dd' dd.dd"
10        d°dd'dd.dd"
11      dd° dd' d.dd"
12       dd° dd'd.dd"
13           dd°dd'd"
14         dd° dd'dd"
15        dd° dd' dd"
dtype: object


========Lat=========
nan
Series([], dtype: object)

invalid format
0          dddddddd
1        dd' dd.dd"
2            dddddd
3    ddd dd' dd.dd"
4            ddd.dd
5

### Parse

Remove trailing non-numbers to make parsing easier

In [16]:
cgg_df.cleaned_lon = cgg_df.cleaned_lon.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')  
cgg_df.cleaned_lat = cgg_df.cleaned_lat.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')

##### Split lon data into degree minute and decimal second

If it looks good apply the splitting to the data

In [17]:
cgg_df['cleaned_lon_split'] = cgg_df.cleaned_lon.str.split(pat=r"[ °']+", regex=True)
cgg_df['cleaned_lat_split'] = cgg_df.cleaned_lat.str.split(pat=r"[ °']+", regex=True)

Manually inspect the splitting

In [18]:
print('============Lon============')
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    
    dms_formats_split = lon_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lon_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

print()
print('============Lat============')
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    
    dms_formats_split = lat_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lat_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

============Lon============
nan

DD
d.dd
['d.dd']
ddd.dd
['ddd.dd']
dd.dd
['dd.dd']

invalid format
dddddddd
['dddddddd']
dd' dd.dd
['dd', 'dd.dd']
dddddd
['dddddd']
'd.dd
['', 'd.dd']
ddddddd
['ddddddd']
dd°dd'ddd
['dd', 'dd', 'ddd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
dd° dddddd
['dd', 'dddddd']

DM
dd°dd.dd
['dd', 'dd.dd']
ddd°dd.dd
['ddd', 'dd.dd']
dd dd.dd
['dd', 'dd.dd']
dd°dd
['dd', 'dd']
ddd°dd
['ddd', 'dd']

DMS
dd°dd'dd
['dd', 'dd', 'dd']
dd°dd'dd.dd
['dd', 'dd', 'dd.dd']
d° dd' dd.dd
['d', 'dd', 'dd.dd']
ddd° dd' dd.dd
['ddd', 'dd', 'dd.dd']
dd dd dd.dd
['dd', 'dd', 'dd.dd']
ddd dd' dd.dd
['ddd', 'dd', 'dd.dd']
ddd dd dd.dd
['ddd', 'dd', 'dd.dd']
ddd°dd'dd.dd
['ddd', 'dd', 'dd.dd']
dd dd' dd
['dd', 'dd', 'dd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
d°dd'dd.dd
['d', 'dd', 'dd.dd']
dd° dd' d.dd
['dd', 'dd', 'd.dd']
dd° dd'd.dd
['dd', 'dd', 'd.dd']
dd°dd'd
['dd', 'dd', 'd']
dd° dd'dd
['dd', 'dd', 'dd']
dd° dd' dd
['dd', 'dd', 'dd']


============Lat============
nan

invalid format

Test if the different formats have correct number of elementes in the split lists and that each element only contains numbers

In [19]:

# =================Lon============================
for i, row in cgg_df[['lon_format', 'cleaned_lon_split']].iterrows():
    split_lst = row.cleaned_lon_split
    lon_format = row.lon_format
    
    if lon_format == 'DD':
        assert len(split_lst) == 1
    if lon_format == 'DM':
        assert len(split_lst) == 2
    if lon_format == 'DMS':
        assert len(split_lst) == 3
        
    if isinstance(split_lst, list) and lon_format != 'invalid format':
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')
            
# =================Lat============================
            
for i, row in cgg_df[['lat_format', 'cleaned_lat_split']].iterrows():
    split_lst = row.cleaned_lat_split
    lat_format = row.lat_format
    
    if lat_format == 'DD':
        assert len(split_lst) == 1
    if lat_format == 'DM':
        assert len(split_lst) == 2
    if lat_format == 'DMS':
        assert len(split_lst) == 3
    if isinstance(split_lst, list):
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')

Convert direction to plus and minus

In [20]:
def convert_direction_lon(row):
    direction = str(row.lon_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[EeØø]', '+', direction)
    direction = re.sub(r'[WwVv]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        
        return 'invalid direction'

def convert_direction_lat(row):
    direction = str(row.lat_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[Nn]', '+', direction)
    direction = re.sub(r'[Ss]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        
        return 'invalid direction'

cgg_df['converted_lon_direction'] = cgg_df.apply(convert_direction_lon, axis=1)
cgg_df['converted_lat_direction'] = cgg_df.apply(convert_direction_lat, axis=1)
print(cgg_df['converted_lat_direction'].unique())
print(cgg_df['converted_lon_direction'].unique())

['' '+' '-' 'invalid direction']
['' '+' '-' 'invalid direction']


Convert Split data into decimal degrees and add converted direction lon

In [21]:
def add_direction(row):
    direction = str(row.converted_lon_direction)
    coord = row.converted_lon
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return np.nan

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
  
    lon_format = row.lon_format 
    split_lst = row.cleaned_lon_split
    
    if lon_format == 'DD':
        result = convert_dd(split_lst)
    elif lon_format == 'DM':
        result = convert_dm(split_lst)
    elif lon_format == 'DMS':
        result = convert_dms(split_lst)
    else:
        return np.nan
    return result
    
cgg_df['converted_lon'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lon'] = cgg_df.apply(add_direction, axis=1)

assert cgg_df.converted_lon.dropna().astype(str).apply(lambda x: bool(re.fullmatch(r'^\-?\d{1,3}\.\d*$', x))).all()

print()
print('The following directions were marked as invalid:')
print(cgg_df.query('converted_lon_direction == "invalid direction"')['lon_direction'].unique())


The following directions were marked as invalid:
['-W' 'N' '-E' 'NW']


Same as above but for lat

In [22]:
def add_direction(row):
    direction = str(row.converted_lat_direction)
    coord = row.converted_lat
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return np.nan

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
    lat_format = row.lat_format 
    split_lst = row.cleaned_lat_split
    if lat_format == 'DD':
        return convert_dd(split_lst)
    elif lat_format == 'DM':
        return convert_dm(split_lst)
    elif lat_format == 'DMS':
        return convert_dms(split_lst)
    else:
        return np.nan
    
cgg_df['converted_lat'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lat'] = cgg_df.apply(add_direction, axis=1)

assert cgg_df.converted_lat.dropna().astype(str).apply(lambda x: bool(re.fullmatch(r'^\-?\d{1,2}\.\d*$', x))).all()

print()
print('The following directions were marked as invalid:')
print(cgg_df.query('converted_lat_direction == "invalid direction"')['lat_direction'].unique())


The following directions were marked as invalid:
['W']


Manually inspect the data to verify lon

In [23]:
cgg_df.query('lon_format == "DMS"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction', 'converted_lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction,converted_lon_direction
10907,"8°27'57.88""E",8°27'57.88,8.466078,DMS,E,+
18972,13° 30′ 36″ E,13° 30' 36,13.510000,DMS,E,+
10679,"46°53'29.17""E",46°53'29.17,46.891436,DMS,E,+
12178,"1° 23' 17.232"" W",1° 23' 17.232,-1.388120,DMS,W,-
3056,N052 26' 51.63'',052 26' 51.63,NaN,DMS,N,invalid direction
3399,N052 26' 51.63'',052 26' 51.63,NaN,DMS,N,invalid direction
13315,"12°34'17.8""E",12°34'17.8,12.571611,DMS,E,+
12501,22° 44' 3.4584'' W,22° 44' 3.4584,-22.734294,DMS,W,-
3031,N052 26' 51.63'',052 26' 51.63,NaN,DMS,N,invalid direction
17614,"13°58'0"" W",13°58'0,-13.966667,DMS,W,-


In [24]:
cgg_df.query('lon_format == "DM"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
2406,37°55′W,37°55,-37.916667,DM,W
5334,113°27'W,113°27,-113.450000,DM,W
5366,113°27'W,113°27,-113.450000,DM,W
3928,E 96º58.478',96°58.478,96.974633,DM,E
10589,015°11.391,015°11.391,15.189850,DM,
5502,113°27'W,113°27,-113.450000,DM,W
5497,113°27'W,113°27,-113.450000,DM,W
4027,E 96º58.478',96°58.478,96.974633,DM,E
5347,113°27'W,113°27,-113.450000,DM,W
5360,113°27'W,113°27,-113.450000,DM,W


In [25]:
cgg_df.query('lon_format == "DD"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
11564,20.3820˚W,20.3820,-20.382000,DD,W
5101,9.846109,9.846109,9.846109,DD,
18883,16.128462,16.128462,16.128462,DD,
8396,22.09,22.09,22.090000,DD,
19624,"-0,940515",0.940515,-0.940515,DD,-
9717,14.4625° E,14.4625,14.462500,DD,E
9684,14.4625° E,14.4625,14.462500,DD,E
2621,-139.538727,139.538727,-139.538727,DD,-
20353,"-19,505538",19.505538,-19.505538,DD,-
11276,89.0706,89.0706,89.070600,DD,


In [26]:
cgg_df.query('lon_format == "invalid format"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
5680,51770889,51770889,NaN,invalid format,
16386,"28' 15.24""",28' 15.24,NaN,invalid format,
20809,"29' 32.522""",29' 32.522,NaN,invalid format,
16349,"28' 15.24""",28' 15.24,NaN,invalid format,
20622,"28' 15.24""",28' 15.24,NaN,invalid format,
16463,29' 32.49'',29' 32.49,NaN,invalid format,
20639,"28' 15.24""",28' 15.24,NaN,invalid format,
5710,8788683,8788683,NaN,invalid format,
16953,19' 29.70'',19' 29.70,NaN,invalid format,
20623,"28' 15.24""",28' 15.24,NaN,invalid format,


Manually inspect the data to verify lat

In [27]:
cgg_df.query('lat_format == "DMS"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
12150,"50° 51' 49.53"" N",50° 51' 49.53,50.863758,DMS,N
12161,"50° 51' 49.53"" N",50° 51' 49.53,50.863758,DMS,N
15659,"44°45'43.73""N",44°45'43.73,44.762147,DMS,N
13331,"45°30'39.37""N",45°30'39.37,45.510936,DMS,N
10863,"71°00'15""N",71°00'15,71.004167,DMS,N
7803,"69˚3'29.35"" N",69°3'29.35,69.058153,DMS,N
10843,"71°00'15""N",71°00'15,71.004167,DMS,N
10684,"34°35'2.19""N",34°35'2.19,34.583942,DMS,N
11549,"50° 51' 49.53"" N",50° 51' 49.53,50.863758,DMS,N
15748,63°46‘27‘‘ N,63°46'27,63.774167,DMS,N


In [28]:
cgg_df.query('lat_format == "DM"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
10569,50° 32.932,50° 32.932,50.548867,DM,
13367,23° 03.131´S,23° 03.131,-23.052183,DM,S
3922,N 73º21.025',73°21.025,73.350417,DM,N
11479,50o31.331,50°31.331,50.522183,DM,
5502,52°04'N,52°04,52.066667,DM,N
5390,52°04'N,52°04,52.066667,DM,N
10625,50° 32.932,50° 32.932,50.548867,DM,
3981,N 73º09.387',73°09.387,73.156450,DM,N
4003,N 73º09.052',73°09.052,73.150867,DM,N
1115,79 43.295 N,79 43.295,79.721583,DM,N


In [29]:
cgg_df.query('lat_format == "DD"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
19559,"59,38289",59.38289,59.382890,DD,
18701,65.69945,65.69945,65.699450,DD,
16636,64.14496,64.14496,64.144960,DD,
10322,3.5947,3.5947,3.594700,DD,
15289,29.88603,29.88603,29.886030,DD,
4168,N53.24889,53.24889,53.248890,DD,N
16707,64.14496,64.14496,64.144960,DD,
14955,30.512656,30.512656,30.512656,DD,
8083,43.18,43.18,43.180000,DD,
16808,55.2894,55.2894,55.289400,DD,


In [30]:
cgg_df.query('lat_format == "invalid format"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
2978,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
20638,63' 15.02'',63' 15.02,NaN,invalid format,
2335,64398201,64398201,NaN,invalid format,
2955,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
16903,61' 56.88'',61' 56.88,NaN,invalid format,
16944,61' 56.88'',61' 56.88,NaN,invalid format,
20773,64' 33.10'',64' 33.10,NaN,invalid format,
3137,W116 11' 47.538'',116 11' 47.538,NaN,invalid format,W
595,+50483248,50483248,NaN,invalid format,+
20798,64' 33.10'',64' 33.10,NaN,invalid format,


### Stats

Number of rows where either lat or lon is invalid or missing

In [31]:
len(cgg_df.query("lat_format == 'invalid format' | converted_lat_direction == 'invalid direction' | lon_format == 'invalid format' | converted_lon_direction == 'invalid direction' | Lat.isnull() | Lon.isnull()"))

9305

In [32]:
cgg_df['data_cleaning_note'] = None
cgg_df.loc[
    (cgg_df["lat_format"] == "invalid format") |
    (cgg_df["converted_lat_direction"] == "invalid direction") |
    (cgg_df["lon_format"] == "invalid format") |
    (cgg_df["converted_lon_direction"] == "invalid direction") |
    (cgg_df["Lat"].isna()) |
    (cgg_df["Lon"].isna()),
    "data_cleaning_note"
] = "Needs human validation"

Save the file

In [33]:
cgg_df.to_csv(r'data/cgg_clean_lat_lon.tsv', sep='\t', index=False)